In [ ]:
pip install langchain_community langchain_openai faiss-cpu pypdf
!pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import fitz  # PyMuPDF
from dotenv import load_dotenv
from openai import OpenAI
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


# .env 파일 로드
load_dotenv()

# API 키 설정
API_KEY = os.getenv("API_KEY")
os.environ['API_KEY'] = API_KEY
client = OpenAI(api_key=API_KEY)

## PDF RAG

In [ ]:
# PDF 파일 읽기 함수
def load_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return text.strip()

# 문서 임베딩 및 벡터화 함수
def create_vector_store(text):
    text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    docs = text_splitter.split_text(text)

    embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key=API_KEY)
    vector_store = FAISS.from_texts(docs, embeddings)
    return vector_store

# PDF 경로 지정
pdf_path = "your_document.pdf"  # 여기에 사용할 PDF 파일 경로 입력

# PDF 로드 및 벡터 저장소 생성
pdf_text = load_pdf_text(pdf_path)
vector_store = create_vector_store(pdf_text)

# RAG 기반 질의 함수
def retrieve_relevant_passages(query, vector_store):
    docs = vector_store.similarity_search(query, k=3)  # 관련 문서 3개 검색
    retrieved_texts = "\n".join([doc.page_content for doc in docs])
    return retrieved_texts

## 답변 생성 함수

In [ ]:
def passage_generate_text(system_prompt, user_prompt, vector_store):
    retrieved_text = retrieve_relevant_passages(user_prompt, vector_store)
    full_prompt = f"참고 정보:\n{retrieved_text}\n\n질문:\n{user_prompt}"
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": full_prompt}
        ]
    )
    return response.choices[0].message.content.strip()

## 시스템 프롬프트

In [7]:
subject_query = "기술"
topic_query = "인공지능과 기계학습"

In [4]:
system_text = '''
당신은 대한민국 대학수학능력시험의 국어 영역 독서 분야 지문을 생성하는 시험출제 전문가이다. 
학문적 주제를 기반으로 공정하고 객관적인 사실을 다루는 지문을 생성해야 한다. 
하나의 주제를 중심으로 지문 전체의 흐름을 유지하면서도, 동일한 내용이나 유사한 논지를 불필요하게 반복하지 말고 각 문장을 논리적으로 전개해야 한다.
단순한 정보 나열보다 개념 간의 관계를 유기적으로 연결하여 논리적으로 서술해야 한다. 

시험을 치르는 수험생이 지문을 읽고 논리적 추론을 수행할 수 있게 작성해야 한다. 
모든 문장은 한국어로 생성하며 문법적으로 완벽해야 한다.'''

## 유저 프롬프트

In [ ]:
passage_text = f'''  
 {subject_query} 분야에서 {topic_query}을 핵심 제재로 활용하여 논리적이고 구조적인 지문을 작성해라.
'''

In [15]:
passage_result = passage_generate_text(system_text, passage_text)

In [16]:
passage_result

'인공지능(AI)과 기계 학습(ML)은 현대 기술 발전의 중심에 있다. 이 두 가지 기술은 여러 분야에서 혁신적인 변화를 주도하고 있다. 먼저, 인공지능은 인간의 지적 능력을 모방하고자 하며, 이를 위해 복잡한 데이터 분석과 의사 결정 과정을 처리한다. 반면, 기계 학습은 대량의 데이터를 훈련하여 패턴을 인식하고 예측 모델을 구축하는 데 초점을 맞춘다. 이 두 기술은 서로 밀접하게 연결되어 있으며, 인공지능의 여러 하위 분야 중 하나로서 기계 학습이 위치한다.\n\n기계 학습의 대표적인 방법에는 지도 학습과 비지도 학습이 있다. 지도 학습은 기존의 레이블이 있는 데이터를 기반으로 하여 새로운 데이터의 결과를 예측하는 데 사용된다. 예를 들어, 고객의 구매 이력을 통해 향후 구매 가능성을 예측하는 시스템이 이에 해당한다. 반대로 비지도 학습은 레이블이 없는 데이터에서 패턴이나 구조를 발견하는 데 목적을 둔다. 이는 고객 세분화를 통해 비슷한 성향의 집단을 나누는 작업에 활용될 수 있다.\n\n좀 더 구체적으로, 지원 벡터 기계(SVM) 나 인공 신경망(ANN)과 같은 기계 학습 알고리즘은 인공지능의 다양한 응용 분야에서 필수적인 역할을 한다. 예를 들어, 의료 영상에서 암 세포를 판별하거나 자율 주행 차량의 경로를 최적화하는 등의 작업에서 활용된다. 이러한 알고리즘은 방대한 데이터를 통해 스스로 학습하고 점점 더 정확한 결과를 도출하도록 설계되었다.\n\n그러나 이러한 기술의 발전에는 여러 가지 도전 과제가 따른다. 첫째, 데이터 편향 문제이다. 인공지능 시스템이 잘못된 데이터나 편향된 데이터를 학습할 경우, 불공정한 결과를 초래할 수 있다. 이 문제를 해결하기 위해서는 데이터 수집 과정에서의 정확성과 다양한 데이터셋의 활용이 필수적이다. 둘째, 개인정보 보호 문제도 주목해야 한다. 인공지능 시스템이 개인 데이터를 처리할 때, 데이터의 익명성을 유지하고 프라이버시를 보호하는 것이 중요한 쟁점으로 떠오르고 있다.\n\n여러 과학자와 연구자들은 인공지능과 기계 학습